# W5 Day1 (05/01 周四): Embedding + 向量检索

## 学习目标
- **理论 (1h)**: 文本 Embedding (BGE/E5); HNSW/IVF/PQ 原理
- **实践 (2h)**: FAISS 实现 HNSW/IVF/PQ + benchmark (召回/速度/内存)
- **产出物**: 向量检索 benchmark notebook

## 核心问题 (面试高频)
1. 什么是文本 Embedding？如何训练一个好的 Embedding 模型？
2. HNSW 的原理是什么？为什么它比 brute-force 快？
3. IVF 和 PQ 分别解决什么问题？可以组合使用吗？
4. 向量检索的召回率和速度怎么平衡？
5. 100 亿向量的检索系统怎么设计？
6. BGE 和 OpenAI Embedding 的区别？什么场景选哪个？

---

## 目录
1. [文本 Embedding](#1)
2. [向量检索概述](#2)
3. [FAISS 索引类型](#3)
4. [HNSW 原理](#4)
5. [IVF + PQ 原理](#5)
6. [Benchmark 实验](#6)
7. [总结 & 思考题](#7)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

TERRACOTTA = '#c2553a'
SAGE = '#5a6b4a'
INK = '#2d2a26'

# FAISS
try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False
    print("faiss 未安装，将用 numpy 模拟")

# Sentence Transformers
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except ImportError:
    HAS_ST = False
    print("sentence-transformers 未安装，将用随机向量模拟")

print(f"FAISS: {'available' if HAS_FAISS else 'not installed'}")
print(f"Sentence-Transformers: {'available' if HAS_ST else 'not installed'}")
print("=" * 60)

---
## 1. 文本 Embedding <a id='1'></a>

### 1.1 什么是文本 Embedding？

将文本映射到固定维度的稠密向量空间，使得**语义相似的文本距离近**。

```
"建筑安全审查" → [0.12, -0.34, 0.56, ..., 0.78]  (768维)
"结构合规检查" → [0.11, -0.32, 0.55, ..., 0.80]  (语义相近 → 余弦相似度高)
"今天天气不错" → [-0.45, 0.67, -0.12, ..., 0.23] (语义不同 → 余弦相似度低)
```

### 1.2 主流 Embedding 模型

| 模型 | 维度 | 特点 | 适用场景 |
|---|---|---|---|
| **BGE (BAAI)** | 768/1024 | 中文优秀，开源 | 中文 RAG |
| **E5 (Microsoft)** | 768/1024 | 多语言，指令式 | 多语言 RAG |
| **GTE (Alibaba)** | 768/1024 | 中文优秀 | 中文 RAG |
| **OpenAI text-embedding-3** | 1536/3072 | 商业 API | 英文为主 |
| **Jina Embeddings** | 768 | 多语言，长文本 | 长文档 |

### 1.3 训练方式

| 方法 | 损失函数 | 数据 |
|---|---|---|
| **对比学习** | InfoNCE / Triplet | (query, positive, negative) 三元组 |
| **蒸馏** | KL 散度 | teacher-student 对 |
| **指令式** | 对比 + 指令前缀 | "query: ..." / "passage: ..." |

### 1.4 关键指标

- **MTEB 排行榜**: 综合评估 Embedding 模型 (检索/分类/聚类/STS)
- **维度**: 768 是主流，384 更快但精度略低
- **最大 Token 数**: 512 (BERT 系) / 8192 (长文本模型)

In [ ]:
# ============ 文本 Embedding 演示 ============

if HAS_ST:
    # 使用 BGE-small (轻量级，适合演示)
    model = SentenceTransformer('BAAI/bge-small-zh-v1.5')
    dim = model.get_sentence_embedding_dimension()
    print(f"模型: BAAI/bge-small-zh-v1.5, 维度: {dim}")
else:
    dim = 384
    print(f"模拟 Embedding (维度: {dim})")

# 测试文本
queries = [
    "建筑安全审查规范",
    "结构合规性检查标准",
    "今天天气真不错",
    "GNN 图神经网络原理",
    "图卷积网络消息传递",
    "Python 数据分析",
]

if HAS_ST:
    embeddings = model.encode(queries, normalize_embeddings=True)
else:
    np.random.seed(42)
    # 模拟语义关系
    base = np.random.randn(len(queries), dim)
    # 让语义相似的更近
    base[1] = base[0] + np.random.randn(dim) * 0.3  # 审查≈检查
    base[4] = base[3] + np.random.randn(dim) * 0.3  # GNN≈图卷积
    embeddings = base / np.linalg.norm(base, axis=1, keepdims=True)

# 计算相似度矩阵
sim_matrix = embeddings @ embeddings.T

# 可视化
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-0.2, vmax=1)
ax.set_xticks(range(len(queries)))
ax.set_yticks(range(len(queries)))
short_labels = [q[:8] for q in queries]
ax.set_xticklabels(short_labels, fontsize=9, rotation=30, ha='right')
ax.set_yticklabels(short_labels, fontsize=9)
for i in range(len(queries)):
    for j in range(len(queries)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Embedding Cosine Similarity Matrix')
plt.colorbar(im)
plt.tight_layout()
plt.show()

print("语义相似对 (余弦相似度 > 0.5):")
for i in range(len(queries)):
    for j in range(i+1, len(queries)):
        if sim_matrix[i,j] > 0.5:
            print(f"  '{queries[i][:10]}' ↔ '{queries[j][:10]}': {sim_matrix[i,j]:.3f}")

---
## 2. 向量检索概述 <a id='2'></a>

### 2.1 为什么需要近似最近邻 (ANN)？

| 方法 | 时间复杂度 | 10亿向量耗时 |
|---|---|---|
| **Brute-force** | O(N×D) | ~小时级 |
| **ANN (HNSW)** | O(log N) | ~毫秒级 |

### 2.2 ANN 方法分类

| 类别 | 方法 | 原理 | 优势 |
|---|---|---|
| **图-based** | HNSW | 多层跳表图 | 高召回、查询快 |
| **聚类-based** | IVF | 先聚类再搜索 | 内存友好 |
| **量化-based** | PQ | 乘积量化压缩 | 极致压缩 |
| **哈希-based** | LSH | 局部敏感哈希 | 速度快但召回低 |
| **混合** | IVF+PQ | 聚类+量化 | 大规模首选 |

### 2.3 评估指标

| 指标 | 含义 |
|---|---|
| **Recall@k** | 前 k 个结果中包含多少真正的最近邻 |
| **QPS** | 每秒查询数 (吞吐量) |
| **内存** | 索引占用的内存 |
| **构建时间** | 建立索引的时间 |

---
## 3. FAISS 索引类型 <a id='3'></a>

### FAISS (Facebook AI Similarity Search)

最流行的向量检索库，支持 CPU 和 GPU。

### 索引类型速查

| 索引 | 类型 | 精确? | 适用规模 | 内存 | 速度 |
|---|---|---|---|---|---|
| `IndexFlatL2` | Brute-force | ✓ | < 100万 | 高 | 慢 |
| `IndexIVFFlat` | IVF | ✗ | < 1000万 | 中 | 中 |
| `IndexIVFPQ` | IVF+PQ | ✗ | < 10亿 | 低 | 快 |
| `IndexHNSW` | HNSW | ✗ | < 1000万 | 高 | 很快 |
| `IndexScalarQuantizer` | SQ | ✗ | < 1亿 | 中 | 快 |

### 选择策略

```
数据量 < 100万  → IndexFlatL2 (精确) 或 IndexHNSW (快)
数据量 < 1000万 → IndexHNSW (召回优先) 或 IndexIVFFlat (内存优先)
数据量 < 10亿  → IndexIVFPQ (平衡)
数据量 > 10亿  → 分布式: IndexIVFPQ + sharding
```

---
## 4. HNSW 原理 <a id='4'></a>

### Hierarchical Navigable Small World

```
Layer 2 (最稀疏):  A -------- D
                   |          |
Layer 1 (中等):    A --- B --- D --- E
                   |   / |   / |   / |
Layer 0 (最密集):  A - B - C - D - E - F
```

**搜索过程**:
1. 从最高层的入口点开始
2. 在当前层贪心搜索最近邻
3. 无法改进时下降到下一层
4. 在最底层找到精确近邻

### HNSW 参数

| 参数 | 含义 | 影响 |
|---|---|---|
| **M** | 每个节点的最大连接数 | M↑ → 召回↑ 内存↑ |
| **efConstruction** | 构建时的搜索范围 | ef↑ → 质量↑ 构建慢↑ |
| **efSearch** | 查询时的搜索范围 | ef↑ → 召回↑ 查询慢↑ |

### 为什么 HNSW 快？

- **跳表结构**: 高层快速"跳"到目标区域，低层精细搜索
- **小世界性质**: 任意两个节点只需少数几步就能到达
- **贪心搜索**: 每步选最近的邻居，快速收敛

---
## 5. IVF + PQ 原理 <a id='5'></a>

### IVF (Inverted File Index)

```
1. 用 K-Means 将向量空间分成 nlist 个簇
2. 查询时先找最近的 nprobe 个簇
3. 只在这 nprobe 个簇中搜索

         簇1     簇2     簇3
        / | \   / | \   / | \
       v1 v2 v3 v4 v5 v6 v7 v8 v9
       
查询 → 找到最近的簇 (nprobe=1) → 只搜索簇2中的向量
```

### PQ (Product Quantization / 乘积量化)

```
原始向量: [0.12, -0.34, 0.56, ..., 0.78]  (768维 × 4字节 = 3072字节)

PQ 压缩:
1. 将 768 维分成 96 个子空间 (每个 8 维)
2. 每个子空间用 K-Means 聚类成 256 个码本
3. 每个子空间只存码本索引 (1字节)

压缩后: 96 字节 (32倍压缩!)
```

### IVF + PQ 组合

1. IVF 先粗筛 → 减少搜索范围
2. PQ 再精排 → 内存高效
3. 适用于 **10亿级** 向量检索

In [ ]:
# ============ Benchmark: 不同索引的性能对比 ============

# 生成数据
np.random.seed(42)
N = 100000  # 10万向量
D = 256     # 256维
N_QUERY = 100
K = 10      # Top-10

xb = np.random.randn(N, D).astype('float32')
xq = np.random.randn(N_QUERY, D).astype('float32')

# Ground truth (brute-force)
if HAS_FAISS:
    flat_index = faiss.IndexFlatL2(D)
    flat_index.add(xb)
    t0 = time.time()
    _, gt = flat_index.search(xq, K)
    gt_time = time.time() - t0
    print(f"Brute-force (ground truth): {gt_time:.4f}s")
else:
    # numpy brute-force
    t0 = time.time()
    gt = np.zeros((N_QUERY, K), dtype=int)
    for i in range(N_QUERY):
        dists = np.sum((xb - xq[i])**2, axis=1)
        gt[i] = np.argsort(dists)[:K]
    gt_time = time.time() - t0
    print(f"Numpy brute-force (ground truth): {gt_time:.4f}s")


def compute_recall(predicted, ground_truth):
    """计算 Recall@K"""
    recalls = []
    for p, g in zip(predicted, ground_truth):
        recalls.append(len(set(p) & set(g)) / len(g))
    return np.mean(recalls)


results = {}

if HAS_FAISS:
    # 1. IVF (nlist=100, nprobe=10)
    quantizer = faiss.IndexFlatL2(D)
    ivf = faiss.IndexIVFFlat(quantizer, D, 100)
    t0 = time.time()
    ivf.train(xb)
    ivf.add(xb)
    build_time = time.time() - t0
    ivf.nprobe = 10
    t0 = time.time()
    _, pred = ivf.search(xq, K)
    query_time = time.time() - t0
    results['IVF (nlist=100)'] = {
        'recall': compute_recall(pred, gt),
        'qps': N_QUERY / query_time,
        'build': build_time,
        'memory': N * D * 4 / 1e6,  # MB (float32)
    }
    
    # 2. IVF+PQ (nlist=100, m=32)
    quantizer2 = faiss.IndexFlatL2(D)
    ivfpq = faiss.IndexIVFPQ(quantizer2, D, 100, 32, 8)
    t0 = time.time()
    ivfpq.train(xb)
    ivfpq.add(xb)
    build_time = time.time() - t0
    ivfpq.nprobe = 10
    t0 = time.time()
    _, pred = ivfpq.search(xq, K)
    query_time = time.time() - t0
    results['IVF+PQ (m=32)'] = {
        'recall': compute_recall(pred, gt),
        'qps': N_QUERY / query_time,
        'build': build_time,
        'memory': N * 32 / 1e6,  # 32 bytes per vector
    }
    
    # 3. HNSW (M=32)
    hnsw = faiss.IndexHNSWFlat(D, 32)
    hnsw.hnsw.efConstruction = 64
    t0 = time.time()
    hnsw.add(xb)
    build_time = time.time() - t0
    hnsw.hnsw.efSearch = 32
    t0 = time.time()
    _, pred = hnsw.search(xq, K)
    query_time = time.time() - t0
    results['HNSW (M=32)'] = {
        'recall': compute_recall(pred, gt),
        'qps': N_QUERY / query_time,
        'build': build_time,
        'memory': N * (D * 4 + 32 * 8) / 1e6,  # 粗估
    }
    
    # 4. Flat (baseline)
    results['Flat (exact)'] = {
        'recall': 1.0,
        'qps': N_QUERY / gt_time,
        'build': 0,
        'memory': N * D * 4 / 1e6,
    }

else:
    # 模拟结果
    results = {
        'Flat (exact)':  {'recall': 1.0,  'qps': 500,    'build': 0,    'memory': 100},
        'IVF (nlist=100)': {'recall': 0.92, 'qps': 5000,   'build': 2.0,  'memory': 100},
        'IVF+PQ (m=32)': {'recall': 0.85, 'qps': 15000,  'build': 8.0,  'memory': 3.2},
        'HNSW (M=32)':   {'recall': 0.98, 'qps': 10000,  'build': 15.0, 'memory': 130},
    }

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
names = list(results.keys())
colors = [INK, SAGE, TERRACOTTA, '#b8860b']

# Recall
ax = axes[0]
recalls = [results[n]['recall'] for n in names]
bars = ax.bar(range(len(names)), recalls, color=colors, alpha=0.85, edgecolor='white')
for i, r in enumerate(recalls):
    ax.text(i, r + 0.01, f'{r:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8, rotation=15)
ax.set_ylabel('Recall@10')
ax.set_title('Recall Comparison')
ax.set_ylim(0.7, 1.05)
ax.grid(True, alpha=0.3, axis='y')

# QPS
ax = axes[1]
qps = [results[n]['qps'] for n in names]
bars = ax.bar(range(len(names)), qps, color=colors, alpha=0.85, edgecolor='white')
for i, q in enumerate(qps):
    ax.text(i, q + max(qps)*0.02, f'{q:.0f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8, rotation=15)
ax.set_ylabel('QPS (queries/sec)')
ax.set_title('Throughput Comparison')
ax.grid(True, alpha=0.3, axis='y')

# Memory
ax = axes[2]
mems = [results[n]['memory'] for n in names]
bars = ax.bar(range(len(names)), mems, color=colors, alpha=0.85, edgecolor='white')
for i, m in enumerate(mems):
    ax.text(i, m + max(mems)*0.02, f'{m:.1f}MB', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8, rotation=15)
ax.set_ylabel('Memory (MB)')
ax.set_title('Memory Usage')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Vector Search Benchmark: {N:,} vectors, dim={D}, top-{K}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 汇总表
print(f"\n{'Index':<20} {'Recall@10':>10} {'QPS':>10} {'Memory':>10} {'Build':>8}")
print("-" * 60)
for n in names:
    r = results[n]
    print(f"{n:<20} {r['recall']:>10.3f} {r['qps']:>10.0f} {r['memory']:>9.1f}M {r['build']:>7.1f}s")

print("\n💡 HNSW: 最佳召回+速度，但内存最高")
print("💡 IVF+PQ: 内存最低，适合大规模部署")
print("💡 选择取决于: 数据规模 × 延迟要求 × 内存预算")

In [ ]:
# ============ HNSW 参数敏感性实验 ============

if HAS_FAISS:
    # 不同 M 值
    m_values = [8, 16, 32, 64]
    m_results = []
    for M in m_values:
        idx = faiss.IndexHNSWFlat(D, M)
        idx.hnsw.efConstruction = 64
        idx.add(xb)
        idx.hnsw.efSearch = 32
        t0 = time.time()
        _, pred = idx.search(xq, K)
        qt = time.time() - t0
        recall = compute_recall(pred, gt)
        m_results.append({'M': M, 'recall': recall, 'qps': N_QUERY / qt})
else:
    m_results = [
        {'M': 8,  'recall': 0.90, 'qps': 15000},
        {'M': 16, 'recall': 0.95, 'qps': 12000},
        {'M': 32, 'recall': 0.98, 'qps': 10000},
        {'M': 64, 'recall': 0.99, 'qps': 7000},
    ]

fig, ax1 = plt.subplots(1, 1, figsize=(8, 5))
ms = [r['M'] for r in m_results]
recalls = [r['recall'] for r in m_results]
qps = [r['qps'] for r in m_results]

ax1.plot(ms, recalls, 'o-', color=TERRACOTTA, linewidth=2, markersize=8, label='Recall@10')
ax1.set_xlabel('M (max connections per node)')
ax1.set_ylabel('Recall@10', color=TERRACOTTA)
ax1.tick_params(axis='y', labelcolor=TERRACOTTA)

ax2 = ax1.twinx()
ax2.plot(ms, qps, 's--', color=SAGE, linewidth=2, markersize=8, label='QPS')
ax2.set_ylabel('QPS', color=SAGE)
ax2.tick_params(axis='y', labelcolor=SAGE)

ax1.set_title('HNSW: M vs Recall/QPS Trade-off')
ax1.grid(True, alpha=0.3)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.show()

print("💡 M=32 通常是最佳平衡: 召回 > 97%, 速度也很快")

---
## 7. 总结 & 思考题 <a id='7'></a>

### 今日核心知识点

1. **文本 Embedding**: 将文本映射到稠密向量空间，BGE/E5/GTE 是主流中文模型
2. **HNSW**: 多层跳表图，O(log N) 搜索，高召回高内存
3. **IVF**: K-Means 聚类分区，先粗筛再精搜
4. **PQ**: 乘积量化压缩，32 倍压缩比
5. **IVF+PQ**: 大规模首选，内存低速度快
6. **Trade-off**: 召回 vs 速度 vs 内存，没有银弹

### 面试高频问题

1. **HNSW 原理？** 多层跳表图 + 贪心搜索 + 小世界性质
2. **IVF 和 PQ 的区别？** IVF 减少搜索范围 (聚类)，PQ 减少存储 (量化)
3. **10 亿向量怎么检索？** IVF+PQ + 分布式 shard
4. **Recall 和速度怎么平衡？** 调 nprobe (IVF) 或 efSearch (HNSW)
5. **Embedding 模型怎么选？** 看 MTEB 排行榜 + 语言支持 + 维度 + 速度

In [ ]:
print("=" * 60)
print("W5 Day1 完成!")
print("=" * 60)
print("""
今日成果:
  ✅ 文本 Embedding 模型对比 (BGE/E5/GTE)
  ✅ 相似度矩阵可视化
  ✅ FAISS 索引对比: Flat / IVF / IVF+PQ / HNSW
  ✅ Recall/QPS/Memory 三维 Benchmark
  ✅ HNSW 参数敏感性实验 (M vs Recall/QPS)

关键结论:
  • HNSW: 召回最高 + 速度最快，但内存最大
  • IVF+PQ: 内存最低，适合 10 亿级部署
  • M=32 是 HNSW 的甜蜜点
  • Embedding 质量决定了检索的上限
""")